# U.S. Federal Regulations — QLoRA Fine-Tuning

**Portable public notebook: no saved outputs or personal paths.**

Select your project environment and open the notebook from the repository root. Install compatible CUDA-enabled PyTorch, Unsloth, Transformers, BitsAndBytes, PEFT, TRL, Datasets, and Hugging Face Hub first. A complete training run remains unverified until curated datasets are ready.

## 1. Configuration

Paths are relative to the repository. Enable the execution flags only for the steps you intend to run. Do not load another model while the baseline or terminal chat occupies the GPU.

In [ ]:
from pathlib import Path

PROJECT = Path.cwd()
if not (PROJECT / 'training_utils.py').is_file():
    raise RuntimeError('Start the notebook from the repository root.')

MODEL_ID = 'unsloth/Qwen3-4B-unsloth-bnb-4bit'
MODEL_DIR = PROJECT / 'models' / 'qwen3-4b'
REVISION_FILE = PROJECT / 'models' / 'qwen3-4b-revision.txt'
TRAIN_FILE = PROJECT / 'data' / 'usgovermenttrain.jsonl'
VALIDATION_FILE = PROJECT / 'data' / 'validation.jsonl'
TEST_FILE = PROJECT / 'data' / 'test.jsonl'
OUTPUT_DIR = PROJECT / 'outputs' / 'qwen3-4b-qlora'
DASHBOARD_URL = 'http://127.0.0.1:8765'

DOWNLOAD_MODEL = False
LOAD_MODEL = True
RUN_TRAINING = True
MAX_SEQ_LENGTH = 512
MAX_STEPS = 10  # Initial smoke test, not the final schedule.
LORA_RANK = 8
SEED = 3407

if RUN_TRAINING and not LOAD_MODEL:
    raise ValueError('Enable LOAD_MODEL before RUN_TRAINING.')


## Run all steps



In [ ]:
from pathlib import Path
from importlib import reload
import training_utils
reload(training_utils)


result = training_utils.run_finetuning(
    project=Path.cwd(),
    kernel_namespace=globals(),
)

print('Saved adapter:', result['adapter_path'])


## 2. Optional checkpoint download

Record the exact revision for reproducibility. Reuse the recorded revision on subsequent downloads. No credentials are embedded in the notebook.

In [ ]:
if DOWNLOAD_MODEL:
    from huggingface_hub import HfApi, snapshot_download
    if REVISION_FILE.exists():
        provenance = dict(line.split('=', 1) for line in REVISION_FILE.read_text().splitlines() if '=' in line)
        if provenance.get('model_id') != MODEL_ID:
            raise ValueError('Model ID does not match recorded provenance.')
        revision = provenance['revision']
    else:
        revision = HfApi().model_info(MODEL_ID).sha
    snapshot_download(repo_id=MODEL_ID, revision=revision, local_dir=str(MODEL_DIR))
    REVISION_FILE.write_text(f'model_id={MODEL_ID}\nrevision={revision}\n', encoding='utf-8')
    print('Checkpoint download complete.')
else:
    print('Download skipped.')


## 3. Curated datasets

Prepare the configured train/validation JSONL files. Each line must have a `messages` list ending in an assistant answer. An initial system message is optional; none is injected. Optional source metadata may accompany the messages.

```json
{"messages":[{"role":"user","content":"A scoped question about a federal provision"},{"role":"assistant","content":"A verified answer preserving conditions and exceptions"}]}
```

The example is a schema illustration, not training content. Keep a separate `data/test.jsonl` for final evaluation. Raw model baseline answers are not training targets. Separate near-duplicates and document/topic groups during curation; exact overlap checks do not establish semantic independence. Configure TRAIN_FILE if your curated file has another name.

In [ ]:
from training_utils import load_examples, check_split_overlap, format_examples

datasets_ready = TRAIN_FILE.is_file() and VALIDATION_FILE.is_file()
if datasets_ready:
    train_examples = load_examples(TRAIN_FILE)
    validation_examples = load_examples(VALIDATION_FILE)
    check_split_overlap(train_examples, validation_examples)
    test_examples = load_examples(TEST_FILE)
    check_split_overlap(train_examples, test_examples)
    check_split_overlap(validation_examples, test_examples)
    print(f'Held-out test examples: {len(test_examples)}')
    print(f'Training examples: {len(train_examples)}')
    print(f'Validation examples: {len(validation_examples)}')
elif RUN_TRAINING:
    raise FileNotFoundError('Prepare curated train and validation datasets first.')
else:
    print('Datasets not prepared yet; training is disabled.')


## 4. Local 4-bit model loading

Import Unsloth before training libraries. Run this cell once per fresh kernel to avoid retaining multiple model instances. Hardware requirements depend on sequence length and available memory.

In [ ]:
if LOAD_MODEL:
    if globals().get('model') is not None:
        raise RuntimeError('A model is already retained. Restart Kernel for a fresh load.')
    import unsloth
    import torch
    from unsloth import FastLanguageModel
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable in the selected environment.')
    if not MODEL_DIR.is_dir() or not list(MODEL_DIR.glob('*.safetensors')):
        raise FileNotFoundError('Download the checkpoint first.')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(MODEL_DIR), max_seq_length=MAX_SEQ_LENGTH,
        dtype=None, load_in_4bit=True, full_finetuning=False, local_files_only=True,
        device_map={'': torch.cuda.current_device()},
    )
    print('Local model loaded.')
else:
    print('GPU loading skipped.')


## 5. Freeze the baseline before training

Use `chat.py` for exploratory conversations and `scripts/ecfr_baseline.py` for the current automated Title 1/2 coverage probes. Run these separately from notebook GPU loading.

Broad section questions and long extractive references are not yet a curated legal QA benchmark. Compare base and adapter on a separate reviewed test set with matching prompts, generation settings, context limits, and source dates. Record truncation and factual errors.

## 6. QLoRA adapter and assistant-only loss

Start with a short smoke test. Reject overlength examples instead of silently truncating the answer. The role markers and target modules below are Qwen3-specific and must be reviewed for another model. Inspect formatting and loss masking before the training cell.

In [ ]:
if RUN_TRAINING:
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only
    train_texts = format_examples(train_examples, tokenizer, MAX_SEQ_LENGTH)
    validation_texts = format_examples(validation_examples, tokenizer, MAX_SEQ_LENGTH)
    if getattr(model, 'peft_config', None):
        raise RuntimeError('An adapter is already attached. Restart Kernel for a fresh run.')
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        bias='none', use_gradient_checkpointing='unsloth', random_state=SEED,
    )
    model.print_trainable_parameters()
    trainer = SFTTrainer(
        model=model, processing_class=tokenizer,
        train_dataset=Dataset.from_dict({'text': train_texts}),
        eval_dataset=Dataset.from_dict({'text': validation_texts}),
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / 'checkpoints'), dataset_text_field='text',
            max_length=MAX_SEQ_LENGTH, packing=False,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            gradient_accumulation_steps=4, max_steps=MAX_STEPS,
            learning_rate=2e-4, warmup_ratio=0.1, optim='adamw_8bit',
            bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
            logging_steps=1, logging_nan_inf_filter=False, include_num_input_tokens_seen='non_padding',
            eval_strategy='steps', eval_steps=MAX_STEPS,
            save_strategy='no', report_to='none', seed=SEED,
        ),
    )
    trainer = train_on_responses_only(
        trainer, instruction_part='<|im_start|>user\n',
        response_part='<|im_start|>assistant\n',
    )
    for split in (trainer.train_dataset, trainer.eval_dataset):
        if any(not any(label != -100 for label in row['labels']) for row in split):
            raise ValueError('An example has no assistant tokens contributing to loss.')
    import monitoring
    from importlib import reload
    reload(monitoring)
    monitor_callback = monitoring.create_training_callback(OUTPUT_DIR / 'monitor', len(train_examples), model, max_steps=MAX_STEPS)
    trainer.add_callback(monitor_callback)
    print(f'Trainer prepared. Dashboard: {DASHBOARD_URL} (ready 0/{MAX_STEPS}).')
else:
    print('Trainer preparation skipped.')


In [ ]:
if RUN_TRAINING:
    if 'monitor_callback' not in globals() or monitor_callback not in trainer.callback_handler.callbacks:
        raise RuntimeError('Prepare the Trainer with its live monitor before training.')
    if trainer.state.global_step > 0:
        raise RuntimeError('This Trainer already ran. Restart Kernel for a fresh run.')
    import json
    from urllib.request import urlopen
    with urlopen(DASHBOARD_URL + '/api/status', timeout=3) as response:
        dashboard_state = json.load(response)['state']
    if dashboard_state.get('run_id') != monitor_callback.log.state['run_id']:
        raise RuntimeError('Dashboard is following a different run. Check its --log-dir.')
    torch.cuda.reset_peak_memory_stats()
    try:
        training_result = trainer.train()
        evaluation_result = next((row for row in reversed(trainer.state.log_history) if 'eval_loss' in row and row.get('step') == trainer.state.global_step), None)
        if evaluation_result is None:
            evaluation_result = trainer.evaluate()
        monitor_callback.set_status('evaluated')
    except BaseException as exc:
        monitor_callback.set_status('interrupted' if isinstance(exc, KeyboardInterrupt) else 'error', error=str(exc))
        raise
    print('Training metrics:', training_result.metrics)
    print('Validation metrics:', evaluation_result)
    print(f'Peak allocated VRAM: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB')
else:
    print('Training skipped.')


## 7. Save adapters and provenance

An adapter requires the same base model; it is not a standalone model. Save artifacts locally under ignored `outputs/`. No automatic upload occurs. Evaluate actual held-out responses before publication; loss alone does not prove factual correctness.

In [ ]:
if RUN_TRAINING:
    import hashlib
    import json
    from importlib.metadata import version
    adapter_dir = OUTPUT_DIR / 'adapter'
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))
    trainer.save_metrics('train', training_result.metrics)
    trainer.save_metrics('eval', evaluation_result)
    run_metadata = {
        'model_id': MODEL_ID,
        'checkpoint_provenance': REVISION_FILE.read_text() if REVISION_FILE.exists() else 'not recorded',
        'max_seq_length': MAX_SEQ_LENGTH, 'max_steps': MAX_STEPS,
        'lora_rank': LORA_RANK, 'seed': SEED, 'enable_thinking': False,
        'dataset_hashes': {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in (TRAIN_FILE, VALIDATION_FILE)},
        'packages': {name: version(name) for name in ('torch', 'unsloth', 'transformers', 'bitsandbytes', 'peft', 'trl', 'datasets')},
    }
    (OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
    print('Adapter and metadata saved under outputs/.')
    if 'monitor_callback' in globals():
        monitor_callback.set_status('saved', adapter_path=str(adapter_dir.relative_to(PROJECT)))
else:
    print('No training artifacts to save.')
